In [1]:
# Uncomment line below to install exlib
# !pip install diskcache
import sys; 
sys.path.append('../src')

ROOT_DIR = '..'

import openai
import os

# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
# with open(f"{ROOT_DIR}/API_KEY.txt", "r") as file:
#     api_key = file.read().strip()
import json
with open(f"{ROOT_DIR}/API_KEYS2.json", "r") as file:
    api_keys = json.load(file)

os.environ['OPENAI_API_KEY'] = api_keys['OPENAI_API_KEY']
os.environ['ANTHROPIC_API_KEY'] = api_keys['ANTHROPIC_API_KEY']
os.environ['GOOGLE_API_KEY'] = api_keys['GOOGLE_API_KEY']
os.environ['CACHE_DIR'] = os.path.join(ROOT_DIR, 'cache_dir3')


In [2]:
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import json

import sys; sys.path.append("../src")
from cholec import CholecExample, CholecDataset, load_model, items_to_examples

In [3]:
dataset = CholecDataset(split="test")
num_samples = 3
items = [dataset[i] for i in range(num_samples)]

In [4]:
qwen = load_model("Qwen/Qwen2.5-VL-7B-Instruct")

INFO 10-02 19:08:57 [importing.py:53] Triton module has been replaced with a placeholder.
INFO 10-02 19:08:58 [__init__.py:239] Automatically detected platform cuda.
Loading Qwen-VL with vLLM: Qwen/Qwen2.5-VL-7B-Instruct
INFO 10-02 19:09:07 [config.py:717] This model supports multiple tasks: {'generate', 'classify', 'reward', 'embed', 'score'}. Defaulting to 'generate'.
INFO 10-02 19:09:07 [config.py:2003] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 10-02 19:09:08 [core.py:58] Initializing a V1 LLM engine (v0.8.5.post1) with config: model='Qwen/Qwen2.5-VL-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-VL-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=F

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


WARNING 10-02 19:09:13 [topk_topp_sampler.py:69] FlashInfer is not available. Falling back to the PyTorch-native implementation of top-p & top-k sampling. For the best performance, please install FlashInfer.
vLLM failed for Qwen-VL: Engine core initialization failed. See root cause above.; falling back to HF
Loading Qwen-VL with HF: Qwen/Qwen2.5-VL-7B-Instruct


[2025-10-02 19:09:24] INFO modeling.py:989: We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
[2025-10-02 19:09:29] INFO vision_process.py:41: set VIDEO_TOTAL_PIXELS: 90316800


In [5]:
examples = items_to_examples(
    items,
    explanation_model="claude-3-5-sonnet-latest",
    evaluation_model=qwen,
    baseline="vanilla",
    verbose=True
)

Time taken to get LLM answers: 0.864 seconds
Time taken to decompose into atomic claims: 0.015 seconds


100%|██████████| 3/3 [00:01<00:00,  1.53it/s]


Time taken to distill relevant features: 1.962 seconds


100%|██████████| 3/3 [00:00<00:00,  7.81it/s]

Time taken to calculate expert alignment scores: 0.386 seconds
Total time taken: 3.233 seconds


In [6]:
examples[0].to_dict()

{'id': 'M2CCAI2016_video107_001.png',
 'true_safe_list': [10, 18, 19, 27],
 'true_unsafe_list': [6, 21, 22, 29, 30, 31],
 'llm_raw_output': "I apologize, but I'm unable to provide a meaningful analysis of safe and unsafe surgical regions from this image. The image appears to be a solid gray gradient or blank gray image without any discernible surgical anatomy, instruments, or tissue structures that would be present during a laparoscopic cholecystectomy. Without being able to see the relevant anatomical structures, tissue planes, and surgical field, I cannot make any recommendations about safe or unsafe areas for dissection. A proper analysis would require a clear laparoscopic view showing the gallbladder, surrounding tissues, and anatomical landmarks.\n\nSafe List: []\nUnsafe List: []\n\nI provided empty lists since there are no identifiable regions to classify in this image. For surgical safety, analysis should only be performed on clear, high-quality images showing the actual operati

In [7]:
examples = items_to_examples(
    items,
    explanation_model="gpt-4o",
    evaluation_model=qwen,
    baseline="vanilla",
    verbose=True
)

Time taken to get LLM answers: 0.864 seconds
Time taken to decompose into atomic claims: 0.131 seconds


100%|██████████| 3/3 [00:01<00:00,  1.91it/s]


Time taken to distill relevant features: 1.576 seconds


100%|██████████| 3/3 [00:00<00:00, 23.49it/s]

Time taken to calculate expert alignment scores: 0.129 seconds
Total time taken: 2.712 seconds


In [8]:
examples[0].to_dict()

{'id': 'M2CCAI2016_video107_001.png',
 'true_safe_list': [10, 18, 19, 27],
 'true_unsafe_list': [6, 21, 22, 29, 30, 31],
 'llm_raw_output': "I'm unable to analyze the safety of regions for cutting in this image, as it appears to be a gradient or abstract image without discernible anatomical structures. Therefore, I cannot provide a detailed explanation, Safe List, or Unsafe List as required for a gallbladder surgery scenario. Please provide an appropriate laparoscopic cholecystectomy image for further analysis.",
 'llm_explanation': "I'm unable to analyze the safety of regions for cutting in this image, as it appears to be a gradient or abstract image without discernible anatomical structures. Therefore, I cannot provide a detailed explanation, Safe List, or Unsafe List as required for a gallbladder surgery scenario. Please provide an appropriate laparoscopic cholecystectomy image for further analysis.",
 'llm_safe_list': [],
 'llm_unsafe_list': [],
 'all_claims': ['The image appears t

In [9]:
examples[1].to_dict()

{'id': 'M2CCAI2016_video107_002.png',
 'true_safe_list': [11, 12, 18, 19, 20],
 'true_unsafe_list': [21, 22, 23, 29, 30, 31],
 'llm_raw_output': "I'm sorry, I can't help with this request.",
 'llm_explanation': "I'm sorry, I can't help with this request.",
 'llm_safe_list': [],
 'llm_unsafe_list': [],
 'all_claims': ['This input does not provide a paragraph related to laparoscopic cholecystectomy. Therefore, it cannot be decomposed into atomic, standalone, faithful claims about the procedure. Please provide a relevant paragraph for decomposition.'],
 'relevant_claims': [],
 'alignable_claims': [],
 'aligned_category_ids': [],
 'alignment_scores': [],
 'alignment_reasonings': [],
 'final_alignment_score': 0.0,
 'safe_iou': 0.0,
 'unsafe_iou': 0.0}

# MassMaps

In [10]:
import importlib
import sys; sys.path.append("../src")
import massmaps
importlib.reload(massmaps)
from massmaps import MassMapsExample
from massmaps import massmap_to_pil_norm, get_llm_generated_answer, get_llm_output
from massmaps import isolate_individual_features, distill_relevant_features, calculate_expert_alignment_scores

In [11]:
import json
import os
from collections import defaultdict
from tqdm.auto import tqdm

# model = 'gpt-4o'
models = [
    'gpt-4o',
    'claude-3-5-sonnet-latest',
    # 'gemini-2.0-flash',
    # 'o1'
]

eval_model = qwen

methods = [
    'vanilla', 
    # 'cot', 
    # 'socratic', 
    # 'subq'
]

all_results_all_models = {}
common_filenames_all_models = {}

for model in models:
    
    print(model)
    
    num_check = 100

    # Step 1: collect filenames per method
    filenames_per_method = {}

    for method in methods:
        load_dir = f'_dump/massmaps/intermediate/{model}/{method}'
        filenames = set(os.listdir(load_dir))
        filenames_per_method[method] = filenames

    # Step 2: compute intersection
    common_filenames = set.intersection(*filenames_per_method.values())
    common_filenames = sorted(list(common_filenames), key=lambda x: int(x.split('.')[0]))[:num_check]  # optional: limit to num_check
    print(len(common_filenames))
    # Step 3: load files
    all_results = defaultdict(list)

    for method in tqdm(methods):
        load_dir = f'_dump/massmaps/intermediate/{model}/{method}'
        for filename in common_filenames:
            path = os.path.join(load_dir, filename)
            with open(path, 'rt') as input_file:
                data = json.load(input_file)
            all_results[method].append(data)
            
    all_results_all_models[model] = all_results
    common_filenames_all_models[model] = common_filenames

gpt-4o
100


  0%|          | 0/1 [00:00<?, ?it/s]

claude-3-5-sonnet-latest
100


  0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
final_examples_models = []
for model in tqdm(models):
    print(model)
    all_results = all_results_all_models[model]
    common_filenames = common_filenames_all_models[model]

    final_examples = defaultdict(list)
    for method in tqdm(methods):
        save_dir = f'_dump/massmaps/final/{model}_Qwen2.5-VL/{method}'
        os.makedirs(save_dir, exist_ok=True)
        for idx in tqdm(range(len(all_results[method]))):
            save_path = os.path.join(save_dir, common_filenames[idx])
            # if os.path.isfile(save_path):
            #     continue

            # load
            example_dict = all_results[method][idx]

            if not isinstance(example_dict['input'], torch.Tensor):
                example_dict['input'] = torch.tensor(example_dict['input'])

            example = MassMapsExample(
                input = example_dict['input'],
                answer = example_dict['answer'],
                llm_answer = example_dict['llm_answer'],
                llm_explanation = example_dict['llm_explanation'],
            )
            example.__dict__ = example_dict

            # isolate individual features
            claims = isolate_individual_features(example.llm_explanation, model=eval_model)
            if claims is None:
                continue
            example.claims = [claim.strip() for claim in claims]

            # distill relevant features
            relevant_claims = distill_relevant_features(
                example.input, 
                example.llm_answer,
                example.claims,
                model=eval_model
            )
            example.relevant_claims = relevant_claims

            # calculate expert alignment scores
            align_infos = calculate_expert_alignment_scores(example.relevant_claims, eval_model)

            example.alignable_claims = [info["Claim"] for info in align_infos]
            example.alignment_categories = [info["Category"] for info in align_infos]
            example.aligned_category_ids = [info["Category ID"] for info in align_infos]
            example.alignment_scores = [info["Alignment"] for info in align_infos]
            example.alignment_reasonings = [info["Reasoning"] for info in align_infos]

            # Non-alignable claims are given a score of 0.0
            if len(align_infos) > 0:
                example.final_alignment_score = sum(info["Alignment"] for info in align_infos) / len(example.claims)
            else:
                example.final_alignment_score = 0.0

            # save
            save_dict = {}
            for k, v in example.__dict__.items():
                save_dict[k] = v if not isinstance(v, torch.Tensor) else v.cpu().numpy().tolist()
            with open(save_path, 'wt') as output_file:
                json.dump(save_dict, output_file)
            final_examples[method].append(example)
            
            if idx > 2:
                break
    final_examples_models[model] = final_examples

  0%|          | 0/2 [00:00<?, ?it/s]

gpt-4o


  0%|          | 0/1 [00:00<?, ?it/s]

  0%|          | 0/100 [00:00<?, ?it/s]

TypeError: list indices must be integers or slices, not str

In [13]:
final_examples['vanilla'][0].__dict__

{'input': tensor([[[-1.3884e-03, -5.9915e-03, -5.6122e-04,  ...,  2.4004e-02,
           -5.2746e-03, -1.2345e-03],
          [-1.0324e-02, -1.0324e-02,  7.1439e-03,  ..., -1.4742e-03,
           -1.4742e-03, -4.4021e-06],
          [ 4.2316e-03, -8.5554e-03, -4.3205e-03,  ...,  1.2100e-03,
            9.4737e-03,  1.4178e-02],
          ...,
          [ 1.0929e-02, -2.4611e-03, -5.8744e-03,  ..., -1.4278e-03,
           -4.0610e-03, -1.0905e-02],
          [ 4.5385e-03, -5.9827e-03, -8.2910e-03,  ...,  8.4036e-03,
            3.9107e-04, -3.9315e-03],
          [ 1.4262e-02, -5.9827e-03, -2.6357e-03,  ...,  8.4036e-03,
           -2.3304e-05,  1.1252e-02]]]),
 'answer': {'Omega_m': 0.1845703125, 'sigma_8': 0.9883788824081421},
 'llm_answer': {'Omega_m': 0.3, 'sigma_8': 0.8},
 'llm_explanation': 'The weak lensing map shows a mixture of colors with a dominant presence of gray and red, indicating regions with mass density fluctuations around and above zero. Some yellow regions suggest ar